# 02. 設定生成 — Visibility の理解

**前提ファイル**: `network.onnx`（`01_onnx.ipynb` または `simple_demo.ipynb` を実行済み）

In [ ]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [ ]:
import ezkl, json, os

model_path    = 'network.onnx'
settings_path = 'settings.json'

assert os.path.exists(model_path), "network.onnx がありません。01_onnx.ipynb を先に実行してください。"

## Visibility とは

ZK 証明では「何を証明者だけが知っていて、何を検証者も知るか」を宣言する必要があります。

| visibility | 意味 | 誰が見える |
|---|---|---|
| `public` | 公開値 | 証明者・検証者の両方 |
| `private` | 秘密値 | 証明者のみ |
| `fixed` | 回路に焼き込み（変更不可） | 回路生成時に決定 |

設定する項目は 3 つ：
- `input_visibility` — 入力データ（画像など）
- `output_visibility` — 推論結果（クラス分類など）
- `param_visibility` — モデルの重み（weight, bias）

## よくある visibility の組み合わせ

| 用途 | input | output | param |
|---|---|---|---|
| 基本デモ（このノートブック） | public | public | fixed |
| 入力を隠したい（プライバシー保護） | private | public | fixed |
| モデルを隠したい（独自モデルの秘匿） | public | public | private |
| 完全秘匿 | private | public | private |

`param = fixed` は「このモデルの重みで計算した」ことを証明するためのコミットメントです。RareSkills の「モデル重みへのコミットメント」に対応します。

In [ ]:
py_run_args = ezkl.PyRunArgs()
py_run_args.input_visibility  = "public"
py_run_args.output_visibility = "public"
py_run_args.param_visibility  = "fixed"

res = ezkl.gen_settings(model_path, settings_path, py_run_args=py_run_args)
print(f"gen_settings: {res}")

## settings.json の中身を確認する

In [ ]:
with open(settings_path) as f:
    s = json.load(f)

print("=== 主要な設定項目 ===")
print(f"input_visibility  : {s['run_args']['input_visibility']}")
print(f"output_visibility : {s['run_args']['output_visibility']}")
print(f"param_visibility  : {s['run_args']['param_visibility']}")
print(f"\n=== 全キー ===")
for k in s.keys():
    print(f"  {k}")

In [ ]:
# run_args の全項目を確認
print("=== run_args の全項目 ===")
for k, v in s['run_args'].items():
    print(f"  {k}: {v}")

## `param_visibility = fixed` vs `private` の違い

```
fixed  → 重みを回路に直接埋め込む
         証明するたびに「この重みで計算した」が保証される
         重みは公開されるが、すり替えは不可能

private → 重みを証明者だけが知る秘密として扱う
          「何らかの重みで正しく計算した」ことだけを証明
          モデルの中身を公開せずに推論を証明できる
```

---
次: `03_calibrate.ipynb` で浮動小数点を ZK 向けの整数に変換します。